# 🖼️ Convolutional Neural Networks (CNNs) — The Complete Deep-Dive

**Topics covered:**
1. Why Dense/MLP networks fail on images
2. What is Convolution — intuition and formal math
3. Kernel vs Filter — disambiguation
4. Step-by-step numerical convolution (matrix trace)
5. Kernel zoo — edge, blur, sharpen visualised on a real image
6. Stride, Padding, and the Output Size Formula
7. Feature Maps — building the activation volume
8. Receptive Field — how CNNs build global understanding
9. Weight Sharing — the efficiency superpower
10. Pooling — Max, Average, Global Average
11. Conv variants — 1×1, Depthwise Separable, Dilated, Transposed
12. Full CNN architecture — design and training end-to-end
13. Batch Normalisation placement in CNNs
14. Parameter count analysis
15. Visualising learned filters and feature maps
16. Senior-level interview Q&A


## 1. 🧱 Why Dense / MLP Networks Fail on Images

### 1.1 The Parameter Explosion Problem
A 224×224 RGB image = **150,528 input values**.  
A first hidden layer with 1,024 neurons in a Dense network needs:
$$150,528 \times 1,024 + 1,024 = \textbf{154 million parameters}$$
just for the first layer — and this is a tiny image by modern standards.

A CNN processes the same image with a single 3×3 filter = **27 parameters**.  
The same filter slides across the entire image (weight sharing) — covering all 150K pixels for 27 weights.

### 1.2 The Position-Blindness Problem
A dense network has a different weight for pixel at (0,0) vs pixel at (100,50).  
If a cat ear moves 10 pixels to the right, the dense network sees a completely different input.  
A CNN uses the same filter everywhere → "cat ear" is detected **wherever it appears**.  
This property is called **Translation Equivariance**.

### 1.3 The No-Spatial-Structure Problem
Dense networks flatten the image to 1D before processing — destroying all spatial relationships.  
A pixel at (5,5) is adjacent to (5,6) and (6,5) — a dense network has no way to encode this.  
Convolution operates on 2D patches — it explicitly uses neighbourhood structure.

| Problem | Dense (MLP) | CNN |
|---|---|---|
| Parameters | 154M for first layer | 27 for a 3×3 filter |
| Position sensitivity | Different weights per position | Same filter everywhere |
| Spatial structure | Destroyed by flattening | Preserved by 2D convolution |
| Translation equivariance | ❌ | ✅ |


## 2. 🔍 What IS Convolution? Intuition + Formal Math

### The Detective Analogy
Imagine you're inspecting a crime scene photo with a **magnifying glass**.
- The **magnifying glass** = the kernel (a small window of learned numbers)
- You **slide it** across the photo pixel by pixel
- At each position, you measure something specific: "Is there an edge here? A texture?"
- You record the result in a new image — the **feature map**

Different lenses detect different features:
- An "edge lens" → highlights boundaries
- A "blur lens" → smooths noise
- A "corner lens" → finds corners

The key insight: the **same lens** is used everywhere in the image.  
This is fundamentally different from a dense layer which uses a different weight at every position.

---

### Formal Definition: Cross-Correlation
In deep learning, "convolution" is technically **cross-correlation** (no kernel flip):

For a 2D input $I$ and kernel $K$ of size $k \times k$, the output $S$ at position $(i, j)$:
$$S(i, j) = \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} I(i+m,\; j+n) \cdot K(m, n)$$

**In words:**
1. Place the kernel over a patch of the input at position $(i, j)$
2. Element-wise multiply each pair
3. Sum all products → write that scalar into $S(i, j)$
4. Slide to the next position and repeat

**True mathematical convolution** would flip the kernel ($K(-m, -n)$) before multiplying.  
DL doesn't flip because the network learns its own weights — it will learn the "flipped" version if needed.

### 3D Extension (for RGB images)
For a colour image with $C$ input channels, the kernel is a 3D tensor of shape $k \times k \times C$:
$$S(i, j) = \sum_{c=0}^{C-1} \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} I_c(i+m, j+n) \cdot K_c(m, n)$$

One filter (3D kernel) still produces **one scalar** per position → **one 2D feature map**.  
Using $F$ filters produces $F$ feature maps → output volume of shape $H' \times W' \times F$.


## 3. 🔢 Step-by-Step Convolution — Numerical Matrix Trace

### Setup
```
Input (5×5):          Kernel (3×3):
1  2  3  0  1         1  0 -1
2  4  1  2  0         1  0 -1
0  1  5  3  1         1  0 -1
1  0  2  4  2
3  1  0  2  1

Stride=1, Padding=0 → Output size = (5-3)/1 + 1 = 3 → Output is 3×3
```

### Computing Output[0,0] — Top-left position
The kernel covers rows 0–2, cols 0–2 of the input:
```
Patch:      Kernel:      Element-wise product:
1  2  3     1  0 -1      1  0 -3
2  4  1  ×  1  0 -1  =   2  0 -1
0  1  5     1  0 -1      0  0 -5

Sum = 1+0+(-3)+2+0+(-1)+0+0+(-5) = -6   → S[0,0] = -6
```

### Computing Output[0,1] — Slide right by stride=1
The kernel now covers rows 0–2, cols 1–3:
```
Patch:      Kernel:      Product:
2  3  0     1  0 -1      2  0   0
4  1  2  ×  1  0 -1  =   4  0  -2
1  5  3     1  0 -1      1  0  -3

Sum = 2+0+0+4+0+(-2)+1+0+(-3) = 2   → S[0,1] = 2
```

### Output[0,2] — Slide right again
Patch: rows 0–2, cols 2–4
```
Sum = (3-1)×1 + (1-0)×0 + (1-(-1)) — let the code compute this
```

**The key point:** The kernel is the **same** every single time — only the patch changes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=2, suppress=True)

# ── Input and Kernel ──
I = np.array([[1,2,3,0,1],
              [2,4,1,2,0],
              [0,1,5,3,1],
              [1,0,2,4,2],
              [3,1,0,2,1]], dtype=float)

K = np.array([[ 1, 0,-1],
              [ 1, 0,-1],
              [ 1, 0,-1]], dtype=float)   # Vertical edge detector (Prewitt)

kH, kW = K.shape
iH, iW = I.shape
oH = iH - kH + 1
oW = iW - kW + 1

# ── Manual convolution (cross-correlation) ──
output = np.zeros((oH, oW))
print("Step-by-step computation (Stride=1, Padding=0):")
print("="*55)
for i in range(oH):
    for j in range(oW):
        patch = I[i:i+kH, j:j+kW]
        product = patch * K
        val = product.sum()
        output[i, j] = val
        if i < 2 and j < 3:   # show first few
            print(f"\nPosition ({i},{j}) — patch rows {i}:{i+kH}, cols {j}:{j+kW}")
            print(f"  Patch:\n{patch}")
            print(f"  Kernel:\n{K}")
            print(f"  Element-wise product:\n{product}")
            print(f"  Sum = {val:.1f}  →  Output[{i},{j}] = {val:.1f}")

print("\n" + "="*55)
print(f"Full Output ({oH}×{oW}):\n{output}")

# ── Visualise ──
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.patch.set_facecolor('#0f0f1a')
titles = ['Input (5×5)', 'Kernel (3×3)
[Vertical Edge Detector]', 'Output Feature Map (3×3)']
data   = [I, K, output]
cmaps  = ['Blues', 'RdBu', 'RdYlGn']

for ax, d, t, cm in zip(axes, data, titles, cmaps):
    ax.set_facecolor('#0f0f1a')
    im = ax.imshow(d, cmap=cm, aspect='auto')
    ax.set_title(t, color='white', fontsize=11)
    for ii in range(d.shape[0]):
        for jj in range(d.shape[1]):
            ax.text(jj, ii, f'{d[ii,jj]:.0f}', ha='center', va='center',
                    color='black', fontsize=10, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("Manual Convolution — Vertical Edge Detection", color='white', fontsize=13)
plt.tight_layout()
plt.savefig('notes/assets/conv_manual.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()


## 4. 🎨 Kernel Zoo — How Different Kernels Affect an Image

Each kernel acts as a different "detector" or "transformer":

| Kernel Name | Purpose | Key Numbers |
|---|---|---|
| **Identity** | No change | Centre=1, all others=0 |
| **Vertical Edge (Prewitt)** | Finds vertical boundaries | Left col=+1, right col=-1 |
| **Horizontal Edge (Prewitt)** | Finds horizontal boundaries | Top row=-1, bottom row=+1 |
| **Sobel X** | Vertical edges (weighted) | Uses 2 in centre column |
| **Laplacian** | All-direction edge detection | Centre=4, neighbours=-1 |
| **Box Blur** | Smooths noise | All values = 1/9 |
| **Gaussian Blur** | Smooth, natural blur | Gaussian distribution weights |
| **Sharpen** | Enhances edges | Centre=5, neighbours=-1 |
| **Emboss** | 3D relief effect | Diagonal gradient pattern |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve

# ── Create synthetic test image with shapes ──
img = np.zeros((60, 60))
# Rectangle (horizontal/vertical edges)
img[10:30, 10:40] = 1.0
# Circle-ish
for i in range(60):
    for j in range(60):
        if (i-45)**2 + (j-45)**2 < 12**2:
            img[i, j] = 0.8
# Gradient stripe
img[35:45, :] = np.linspace(0, 1, 60)

kernels = {
    'Identity':         np.array([[0,0,0],[0,1,0],[0,0,0]]),
    'Vertical Edge':    np.array([[-1,0,1],[-1,0,1],[-1,0,1]]),
    'Horizontal Edge':  np.array([[-1,-1,-1],[0,0,0],[1,1,1]]),
    'Sobel X':          np.array([[-1,0,1],[-2,0,2],[-1,0,1]]),
    'Laplacian':        np.array([[0,-1,0],[-1,4,-1],[0,-1,0]]),
    'Box Blur':         np.ones((3,3)) / 9,
    'Sharpen':          np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]),
    'Emboss':           np.array([[-2,-1,0],[-1,1,1],[0,1,2]]),
}

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle("Kernel Zoo — Same Image, Different Filters", color='white', fontsize=14)

for ax, (name, k) in zip(axes.flat, kernels.items()):
    ax.set_facecolor('#0f0f1a')
    filtered = convolve(img, k)
    filtered = np.clip(filtered, filtered.min(), filtered.max())
    ax.imshow(filtered, cmap='gray', vmin=filtered.min(), vmax=filtered.max())
    ax.set_title(f'{name}\nKernel: {k.tolist()}', color='#4ecdc4', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('notes/assets/kernel_zoo.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Saved → notes/assets/kernel_zoo.png")


## 5. 📐 Stride, Padding & the Output Size Formula

### The Master Formula
$$H_{out} = \left\lfloor \frac{H_{in} + 2P - D(K-1) - 1}{S} + 1 \right\rfloor$$

where:
- $H_{in}$ = input height (same formula applies for width $W$)
- $P$ = padding (zeros added around each edge)
- $K$ = kernel size
- $S$ = stride (step size)
- $D$ = dilation factor (1 for standard convolution)

### Padding Modes
| Mode | Zeros Added | Effect | Use Case |
|---|---|---|---|
| **Valid (no padding)** | $P=0$ | Output shrinks by $K-1$ each layer | When you want aggressive downsampling |
| **Same padding** | $P=(K-1)/2$ | Output = same size as input (stride=1) | Most common — preserves spatial dims |
| **Full padding** | $P=K-1$ | Output larger than input | Rarely used in DL |

### Stride Effect
| Stride | Effect | Result |
|---|---|---|
| $S=1$ | Move 1 pixel per step | Dense output — preserves size (with same padding) |
| $S=2$ | Move 2 pixels per step | Output ~half the size — replaces pooling |
| $S=3+$ | Aggressive skip | Very coarse output |

### Worked Examples

**Example 1:** Input=32×32, K=3, P=1, S=1
$$H_{out} = \frac{32 + 2(1) - 3}{1} + 1 = \frac{31}{1} + 1 = 32 ✅ \text{ (same padding!)}$$

**Example 2:** Input=224×224, K=3, P=1, S=2
$$H_{out} = \frac{224 + 2(1) - 3}{2} + 1 = \frac{223}{2} + 1 = 111 + 1 = 112$$
Output is 112×112 — the standard ResNet stem halving.

**Example 3:** Input=7×7, K=7, P=0, S=1 (Global-style conv)
$$H_{out} = \frac{7 + 0 - 7}{1} + 1 = 0 + 1 = 1$$
Output is 1×1 — collapses spatial dims entirely.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def conv_output_size(H_in, K, P, S, D=1):
    return int(np.floor((H_in + 2*P - D*(K-1) - 1) / S + 1))

# ── Visualise how stride & padding affect output size ──
configs = [
    ('Valid, S=1', 3, 0, 1),
    ('Same, S=1',  3, 1, 1),
    ('Same, S=2',  3, 1, 2),
    ('Large K, S=1, Same', 5, 2, 1),
    ('Large K, S=2',  5, 2, 2),
]

print(f"{'Config':<25} {'H_in':>6} → {'H_out':>6}  (K, P, S)")
print("-" * 55)
for H in [28, 56, 112, 224]:
    for name, K, P, S in configs:
        H_out = conv_output_size(H, K, P, S)
        print(f"  {name:<23} {H:>5} → {H_out:>5}   (K={K}, P={P}, S={S})")
    print()

# ── Visual: stride effect on a 1D signal ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.patch.set_facecolor('#0f0f1a')
signal = np.sin(np.linspace(0, 4*np.pi, 60)) + 0.3*np.random.randn(60)
kernel = np.array([1/3, 1/3, 1/3])  # box blur

for ax, (stride, title, col) in zip(axes, [
    (1, 'Stride = 1
(dense output)', '#4ecdc4'),
    (2, 'Stride = 2
(half resolution)', '#fd79a8'),
    (3, 'Stride = 3
(coarse output)', '#fdcb6e'),
]):
    ax.set_facecolor('#13131f')
    conv_out = np.convolve(signal, kernel, 'valid')[::stride]
    x_in  = np.arange(len(signal))
    x_out = np.arange(len(conv_out)) * stride

    ax.plot(x_in, signal, color='#aaa', lw=1, alpha=0.6, label='Input signal')
    ax.plot(x_out, conv_out, 'o-', color=col, lw=2, ms=4, label=f'Output ({len(conv_out)} pts)')
    ax.set_title(title, color='white')
    ax.set_xlabel('Position', color='#aaa')
    ax.legend(fontsize=9)
    ax.tick_params(colors='#aaa')
    for sp in ax.spines.values(): sp.set_color('#333')

plt.suptitle("Effect of Stride on Output Resolution (1D Analogy)", color='white', fontsize=13)
plt.tight_layout()
plt.savefig('notes/assets/stride_effect.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()


## 6. 🗺️ Feature Maps, Receptive Field & Weight Sharing

### Feature Maps
When $F$ filters each slide over the input:
- Each filter produces one **feature map** (2D)
- $F$ filters → $F$ feature maps → stacked into a 3D **activation volume** of shape $H' \times W' \times F$

**Intuition:** If F=64 at layer 1:
- Feature map 1: "Where are vertical edges?"
- Feature map 2: "Where are horizontal edges?"
- Feature map 3: "Where are diagonal lines?"
- … 
- Feature map 64: "Where is some complex edge+texture combination?"

---

### Receptive Field — How CNNs Build Global Understanding
The **receptive field** of a neuron is the region of the original input image that influences its activation.

**Layer 1 neuron** with 3×3 kernel: sees a **3×3** patch of the input.

**Layer 2 neuron** with 3×3 kernel: each of its 3×3 input neurons already saw 3×3 → effective receptive field = **5×5**.

**Layer 3:** **7×7**. After $L$ layers with 3×3 kernels: receptive field = $2L+1$.

For VGG-16 (13 conv layers with 3×3): RF = **27×27**
For AlexNet (with 11×11 first kernel): even larger.

**Why this matters:** Deep CNNs can capture global patterns (entire faces, full objects) even though each individual kernel only looks at a tiny local patch.

---

### Weight Sharing — The Efficiency Superpower
In a **Dense layer**: every output neuron has its own unique weight vector → $H \times W \times C$ different weights per neuron.

In a **Conv layer**: every position in the output uses the **same** filter weights → only $K \times K \times C_{in}$ unique weights per filter.

For 224×224 input with 64 neurons per position in a dense layer:
- Dense: $224 \times 224 \times 3 \times 64 = 9.7M$ weights (for just 64 neurons!)
- Conv (3×3): $3 \times 3 \times 3 \times 64 = 1,728$ weights

**Benefits of weight sharing:**
1. **Parameter efficiency** — train far fewer parameters
2. **Translation equivariance** — a cat ear is detected wherever it appears
3. **Generalisation** — the filter learns a general feature detector, not position-specific memorisation


## 7. 🏊 Pooling — Compressing the Feature Volume

### What Is Pooling?
After convolution, the feature maps are still large. A **Pooling layer** summarises each region into a single value — like asking your team for just the key highlights rather than every detail.

Pooling has **no learnable parameters** — it's a fixed mathematical operation.

### Max Pooling (Industry Standard)
Takes the **maximum** value in each window:
```
Input (4×4):       Pool window (2×2, stride=2):
12  20  30   0     Top-left:   max(12,20,8,12) = 20
 8  12   2   0     Top-right:  max(30,0,2,0)   = 30
34  70  37   4     Bot-left:   max(34,70,112,100) = 112
112 100  25  12    Bot-right:  max(37,4,25,12)  = 37

Output (2×2):
20   30
112  37
```

**Why max works:** Image features (edges, textures) are represented by their **strongest activation**. Max pooling keeps the strongest signal and discards weak noise.

### Average Pooling
Takes the **mean** of the window. Rarely used in hidden layers (dilutes strong activations).  
Still used in some smoothing architectures.

### Global Average Pooling (GAP) — Modern Standard
Takes the average of the **entire feature map** (no window size, no stride):
- Input: $H \times W \times C$ feature volume
- Output: a vector of length $C$ (one number per channel)
- **Replaces Flatten + Dense** in modern architectures (ResNet, MobileNet)

**Why GAP > Flatten:**
- Flatten → Dense: adds millions of parameters. A 7×7 feature map with 512 channels → 7×7×512 = 25,088 → dense(1000) = 25M parameters
- GAP: zero learnable parameters, forces meaningful spatial learning
- GAP also makes the model resolution-agnostic (works on any image size)

### Comparison Table
| Property | Max Pool | Avg Pool | GAP |
|---|---|---|---|
| Keeps strongest signal | ✅ | ❌ | ❌ |
| Noise suppression | ✅ | ⚠️ | ✅ |
| Learnable params | 0 | 0 | 0 |
| Output size | $H/2 \times W/2$ | $H/2 \times W/2$ | $1 \times 1$ per channel |
| Standard use | Hidden layers | Specific cases | Final layer (modern CNNs) |


In [ ]:
import torch, torch.nn as nn
import numpy as np, matplotlib.pyplot as plt

# ── Manual pool visualisation ──
fm = np.array([[12, 20, 30,  0],
               [ 8, 12,  2,  0],
               [34, 70, 37,  4],
               [112,100, 25, 12]], dtype=float)

def max_pool2d(x, size=2, stride=2):
    H, W = x.shape
    out = np.zeros((H//stride, W//stride))
    for i in range(0, H-size+1, stride):
        for j in range(0, W-size+1, stride):
            out[i//stride, j//stride] = x[i:i+size, j:j+size].max()
    return out

def avg_pool2d(x, size=2, stride=2):
    H, W = x.shape
    out = np.zeros((H//stride, W//stride))
    for i in range(0, H-size+1, stride):
        for j in range(0, W-size+1, stride):
            out[i//stride, j//stride] = x[i:i+size, j:j+size].mean()
    return out

max_out = max_pool2d(fm)
avg_out = avg_pool2d(fm)
gap_val = fm.mean()

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle("Pooling Types — Numerical Comparison", color='white', fontsize=13)

for ax, data, title, cmap in zip(
    axes,
    [fm, max_out, avg_out],
    ['Input Feature Map (4×4)', 'Max Pooling Output (2×2)', 'Avg Pooling Output (2×2)'],
    ['Blues', 'Greens', 'Oranges']
):
    ax.set_facecolor('#0f0f1a')
    im = ax.imshow(data, cmap=cmap)
    ax.set_title(title, color='white', fontsize=11)
    ax.axis('off')
    for ii in range(data.shape[0]):
        for jj in range(data.shape[1]):
            ax.text(jj, ii, f'{data[ii,jj]:.0f}', ha='center', va='center',
                    color='black', fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig('notes/assets/pooling_types.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

print(f"Input:      {fm.shape}  values range: {fm.min():.0f} → {fm.max():.0f}")
print(f"Max Pool:   {max_out.shape}  max preserved: {max_out.max():.0f}")
print(f"Avg Pool:   {avg_out.shape}  smoothed:       {avg_out.max():.0f}")
print(f"GAP output: scalar = {gap_val:.1f}")
print("\nNote: Max Pool keeps 112 (the dominant feature).")
print("Avg Pool returns 79 (the strong signal is diluted by surrounding zeros).")


## 8. 🔀 Convolution Variants

### 8.1 1×1 Convolution — The Bottleneck
A 1×1 conv looks at **one pixel across all channels** — it's a learned linear projection across the depth dimension.

**Why is it useful?**
1. **Channel reduction (bottleneck):** 512 channels → 64 channels with 64 × (1×1×512) = 32,768 params
   Compare to a 3×3 conv directly: 512 channels → 512 with (3×3×512)×512 = 2.4M params
2. **Add non-linearity without spatial change:** Apply ReLU after a 1×1 conv → complexity for free
3. **Used in ResNet blocks, MobileNet, GoogLeNet/Inception**

---

### 8.2 Depthwise Separable Convolution
Standard convolution mixes spatial filtering + channel mixing into one operation.  
**Depthwise Separable** splits this into two steps:

**Step 1 — Depthwise conv:** One kernel per channel (spatial only, no cross-channel mixing)  
**Step 2 — Pointwise conv (1×1):** Mix channels  

**Parameter count comparison (3×3 kernel, C_in=C_out=256):**
- Standard: $3 \times 3 \times 256 \times 256 = 589,824$
- Depthwise sep.: $3 \times 3 \times 256 + 1 \times 1 \times 256 \times 256 = 2,304 + 65,536 = 67,840$
- **~8.7× fewer parameters!** Used in MobileNet, Xception

---

### 8.3 Dilated (Atrous) Convolution
Instead of applying the kernel to adjacent pixels, **skip** $D-1$ pixels between kernel elements (dilation rate $D$).

A 3×3 kernel with dilation $D=2$ covers a **5×5** area with only 9 parameters!

$$H_{out} = \frac{H_{in} + 2P - D(K-1) - 1}{S} + 1$$

**Why it's powerful:**
- Exponentially expands receptive field without adding parameters
- Used in **semantic segmentation** (DeepLab), **speech synthesis** (WaveNet)
- Stacking dilations [1, 2, 4, 8] covers a 15×15 effective field with 4 small kernels

---

### 8.4 Transposed Convolution (ConvTranspose)
Also called "deconvolution" — used to **upsample** feature maps (go from small → large).

Used in:
- **Autoencoders** (decoder path)
- **GANs** (generator)
- **Semantic segmentation** (FCN, U-Net decoder)

Not the mathematical inverse of convolution — it's a learned upsampling.  
Stride > 1 in a transposed conv = upsampling factor.


In [ ]:
import torch, torch.nn as nn

# ── 1×1 Convolution — Channel Bottleneck ──
print("=" * 55)
print("1×1 Convolution — Bottleneck")
print("=" * 55)
x = torch.randn(1, 512, 14, 14)   # batch=1, C=512, H=14, W=14
conv1x1 = nn.Conv2d(512, 64, kernel_size=1)  # reduce 512→64 channels
out = conv1x1(x)
params = sum(p.numel() for p in conv1x1.parameters())
print(f"Input:  {tuple(x.shape)}")
print(f"Output: {tuple(out.shape)}")
print(f"Params: {params:,}  (vs {3*3*512*512:,} for a 3×3 conv same channels)")

# ── Depthwise Separable Convolution ──
print("\n" + "=" * 55)
print("Depthwise Separable Convolution")
print("=" * 55)
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, C_in, C_out, K=3, padding=1):
        super().__init__()
        # Step 1: Depthwise (one filter per channel, groups=C_in)
        self.depthwise  = nn.Conv2d(C_in, C_in, K, padding=padding, groups=C_in)
        # Step 2: Pointwise (1×1 cross-channel mixing)
        self.pointwise  = nn.Conv2d(C_in, C_out, 1)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

dsc = DepthwiseSeparableConv(256, 256)
std_conv = nn.Conv2d(256, 256, 3, padding=1)
x256 = torch.randn(1, 256, 28, 28)

dsc_params  = sum(p.numel() for p in dsc.parameters())
std_params  = sum(p.numel() for p in std_conv.parameters())
print(f"Standard 3×3 Conv (256→256) params: {std_params:,}")
print(f"Depthwise Separable     (256→256) params: {dsc_params:,}")
print(f"Parameter savings: {(1 - dsc_params/std_params):.1%}")
print(f"Output shape (DSC): {tuple(dsc(x256).shape)}")

# ── Dilated Convolution ──
print("\n" + "=" * 55)
print("Dilated Convolution — Expanded Receptive Field")
print("=" * 55)
for dilation in [1, 2, 4]:
    eff_rf = (3-1)*dilation + 1
    conv_d = nn.Conv2d(1, 1, 3, padding=dilation, dilation=dilation)
    params_d = sum(p.numel() for p in conv_d.parameters())
    print(f"  Dilation={dilation}  → Effective receptive field: {eff_rf}×{eff_rf}  | Params: {params_d}")

# ── Transposed Convolution ──
print("\n" + "=" * 55)
print("Transposed Convolution — Upsampling")
print("=" * 55)
small = torch.randn(1, 64, 7, 7)
upconv = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
large  = upconv(small)
params_up = sum(p.numel() for p in upconv.parameters())
print(f"Input:  {tuple(small.shape)}")
print(f"Output: {tuple(large.shape)}  (7×7 → 14×14, stride=2 upsamples by 2×)")
print(f"Params: {params_up:,}")


## 9. 🏛️ Full CNN Architecture — Design Principles

### The Standard Backbone Pattern
Most CNN architectures follow this progression:
```
Input Image (H×W×C)
    ↓
[Conv → BN → ReLU] × N   ← feature extraction block
    ↓ (optional Pooling / stride=2)
[Conv → BN → ReLU] × N   ← deeper block (more channels, smaller spatial)
    ↓ (optional Pooling / stride=2)
... repeat ...
    ↓
Global Average Pooling    ← collapse spatial dims
    ↓
Dense → Softmax / Sigmoid ← classification head
```

### Channel Growth Convention
Channels typically **double** as spatial dimensions **halve**:
- Layer 1: 32 channels, 224×224
- Layer 2: 64 channels, 112×112
- Layer 3: 128 channels, 56×56
- Layer 4: 256 channels, 28×28

**Why?** We want to roughly preserve the total amount of information (H × W × C).

### Where to Place Batch Normalisation
The standard order: **Conv → BN → ReLU**  
(Some papers try ReLU → BN, but BN → ReLU is the dominant empirical choice.)

BN in CNNs normalises across the batch AND spatial dimensions (H, W) for each channel.  
It gives every layer a stable distribution of activations → faster, more stable training.

### Parameter Count Analysis (VGG-style small CNN)
```
Layer 1: Conv(3→32, K=3, P=1)  params = 3×3×3×32 + 32 = 896
Layer 2: Conv(32→64, K=3, P=1) params = 3×3×32×64 + 64 = 18,496
Layer 3: Conv(64→128, K=3, P=1) params = 3×3×64×128 + 128 = 73,856
GAP: 0 params
FC(128→10): 128×10 + 10 = 1,290
Total: ~94,000 params — tiny but effective!
```


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(42)

# ── Small CNN for CIFAR-10-style classification ──
class SmallCNN(nn.Module):
    """Parameter-efficient CNN with modern design choices:
    - Conv → BN → ReLU blocks
    - Stride-2 conv instead of MaxPool (learns to downsample)
    - Global Average Pooling instead of Flatten
    - Residual-lite skip for depth
    """
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        # Block 1: 3 × H × W → 32 × H × W
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        # Block 2: 32 → 64, downsample via stride=2
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
        )
        # Block 3: 64 → 128, downsample
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.gap  = nn.AdaptiveAvgPool2d(1)   # Global Average Pooling → 1×1
        self.drop = nn.Dropout(0.4)
        self.fc   = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.block1(x)  # 32×H×W
        x = self.block2(x)  # 64×H/2×W/2
        x = self.block3(x)  # 128×H/4×W/4
        x = self.gap(x)     # 128×1×1
        x = x.flatten(1)    # 128
        x = self.drop(x)
        return self.fc(x)   # num_classes

model = SmallCNN()
x_batch = torch.randn(4, 3, 32, 32)   # 4 images, 32×32 RGB
out = model(x_batch)

# ── Print architecture with shapes ──
print("SmallCNN — Layer-by-Layer Shape Trace")
print("=" * 50)
shapes = {}
x_ = torch.randn(1, 3, 32, 32)
hooks = []
layer_names = []

def hook_fn(name):
    def fn(m, inp, out): shapes[name] = tuple(out.shape)
    return fn

for name, layer in [('block1', model.block1), ('block2', model.block2),
                     ('block3', model.block3), ('gap', model.gap)]:
    hooks.append(layer.register_forward_hook(hook_fn(name)))

with torch.no_grad(): model(x_)
for h in hooks: h.remove()

print(f"  Input:         {(1,3,32,32)}")
for name in ['block1', 'block2', 'block3', 'gap']:
    s = shapes[name]
    print(f"  After {name:<8}: {s}")
print(f"  After flatten: (1, 128)")
print(f"  After fc:      (1, 10)")

total = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total:,}")
print(f"Output shape: {tuple(out.shape)}  (4 images × 10 classes)")

# ── Per-layer parameter breakdown ──
print("\nPer-layer parameter count:")
for name, m in model.named_modules():
    p = sum(p.numel() for p in m.parameters(recurse=False))
    if p > 0:
        print(f"  {name:<30} {p:>8,} params")


## 10. 🛠️ End-to-End Training with Visualisations

In [ ]:
import torch, torch.nn as nn, matplotlib.pyplot as plt
torch.manual_seed(42)

# ── Synthetic image dataset (CIFAR-10-style, 2-class for speed) ──
N, C, H, W = 400, 3, 32, 32
# Class 0: mostly low-frequency patterns
X0 = torch.randn(N//2, C, H, W) * 0.5 + torch.sin(torch.linspace(0,6,H*W)).view(H,W)
# Class 1: mostly high-frequency (noise-like)
X1 = torch.randn(N//2, C, H, W)
X  = torch.cat([X0, X1])
y  = torch.cat([torch.zeros(N//2), torch.ones(N//2)]).long()
perm = torch.randperm(N); X, y = X[perm], y[perm]
X_tr, y_tr = X[:300], y[:300]
X_va, y_va = X[300:], y[300:]

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(64, 2)
    def forward(self, x):
        return self.fc(self.net(x).flatten(1))

model  = TinyCNN()
opt    = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched  = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-2,
                                              steps_per_epoch=10, epochs=50)
loss_fn= nn.CrossEntropyLoss()

tr_losses, va_accs = [], []
BATCH = 32

for epoch in range(50):
    model.train(); ep_loss = 0
    perm = torch.randperm(300)
    for b in range(0, 300, BATCH):
        idx = perm[b:b+BATCH]; xb, yb = X_tr[idx], y_tr[idx]
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); ep_loss += loss.item()
    tr_losses.append(ep_loss / (300//BATCH))
    model.eval()
    with torch.no_grad():
        va_accs.append((model(X_va).argmax(1)==y_va).float().mean().item())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#0f0f1a')
for ax in axes: ax.set_facecolor('#13131f')

axes[0].plot(tr_losses, color='#4ecdc4', lw=2, label='TrainLoss')
axes[0].fill_between(range(len(tr_losses)), tr_losses, alpha=0.15, color='#4ecdc4')
axes[0].set_title('Training Loss (CrossEntropy)', color='white')
axes[0].set_xlabel('Epoch', color='#aaa'); axes[0].legend()
axes[0].tick_params(colors='#aaa')

axes[1].plot(va_accs, color='#fd79a8', lw=2, label='Val Accuracy')
axes[1].fill_between(range(len(va_accs)), va_accs, alpha=0.15, color='#fd79a8')
axes[1].axhline(0.5, color='#fdcb6e', ls='--', lw=1, label='Random baseline')
axes[1].set_title(f'Validation Accuracy (Final: {va_accs[-1]:.1%})', color='white')
axes[1].set_xlabel('Epoch', color='#aaa'); axes[1].legend()
axes[1].tick_params(colors='#aaa')

for ax in axes:
    for sp in ax.spines.values(): sp.set_color('#333')

plt.suptitle("TinyCNN — End-to-End Training", color='white', fontsize=13)
plt.tight_layout()
plt.savefig('notes/assets/cnn_training.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print(f"Final val accuracy: {va_accs[-1]:.1%}  |  Params: {sum(p.numel() for p in model.parameters()):,}")


## 11. 🔬 Visualising Learned Filters & Feature Maps

In [ ]:
import torch, matplotlib.pyplot as plt
import numpy as np

# Use the trained TinyCNN from above (re-instantiate + show filter grids)
model.eval()

# ── Visualise first-layer filters ──
w = model.net[0].weight.data.cpu()   # (16, 3, 3, 3)
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle("Learned First-Layer Filters (16 filters, 3×3 each)", color='white', fontsize=13)

for idx, ax in enumerate(axes.flat):
    ax.set_facecolor('#0f0f1a')
    if idx < w.shape[0]:
        filt = w[idx].permute(1, 2, 0).numpy()
        filt = (filt - filt.min()) / (filt.max() - filt.min() + 1e-8)
        ax.imshow(filt)
        ax.set_title(f'Filter {idx+1}', color='#aaa', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('notes/assets/cnn_filters.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

# ── Visualise feature maps for one test image ──
test_img = X_va[0:1]   # 1 image
activations = {}

def get_activation(name):
    def hook(m, inp, out): activations[name] = out.detach()
    return hook

h1 = model.net[0].register_forward_hook(get_activation('conv1'))
h2 = model.net[3].register_forward_hook(get_activation('conv2'))

with torch.no_grad(): model(test_img)
h1.remove(); h2.remove()

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle("Feature Maps After Conv1 (first 16 channels)", color='white', fontsize=13)
fmaps = activations['conv1'].squeeze()
for idx, ax in enumerate(axes.flat):
    ax.set_facecolor('#0f0f1a')
    if idx < fmaps.shape[0]:
        ax.imshow(fmaps[idx].numpy(), cmap='inferno')
        ax.set_title(f'Map {idx+1}', color='#aaa', fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig('notes/assets/cnn_feature_maps.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Each feature map highlights what a specific filter 'detected' in the input image.")


## 12. 📋 CNN Design Decisions — Reference Tables

### Receptive Field Quick Reference (3×3 kernels, stride=1)
| Depth (layers) | Receptive field |
|---|---|
| 1 | 3×3 |
| 2 | 5×5 |
| 3 | 7×7 |
| 5 | 11×11 |
| 10 | 21×21 |
| 13 (VGG-16) | 27×27 |

### Parameter Count Formula Summary
| Layer type | Formula |
|---|---|
| Conv ($K$×$K$, $C_{in}$→$C_{out}$) | $K^2 \cdot C_{in} \cdot C_{out} + C_{out}$ (with bias) |
| Depthwise Conv ($K$×$K$, $C$ channels) | $K^2 \cdot C + C$ |
| Pointwise (1×1, $C_{in}$→$C_{out}$) | $C_{in} \cdot C_{out} + C_{out}$ |
| BatchNorm ($C$ channels) | $2C$ (gamma, beta) |
| MaxPool / AvgPool | 0 |
| GAP | 0 |
| Dense ($n_{in}$→$n_{out}$) | $n_{in} \cdot n_{out} + n_{out}$ |

---

## 13. 🎯 Senior-Level Interview Q&A

**Q1: Why is "same padding" preferred in most hidden layers?**  
> Without padding (valid), each conv layer reduces spatial dimensions by $K-1$ per side. For a 32×32 image with 10 conv layers of K=3, the spatial dimension shrinks by 2 each layer: 32→30→28→…→12. You lose information and cannot go deep. Same padding keeps spatial dims constant, allowing arbitrary depth while only changing resolution via explicit stride/pooling.

---

**Q2: What is the advantage of using stride-2 convolution over MaxPooling for downsampling?**  
> MaxPooling is a fixed operation — it always picks the max and discards the rest. A stride-2 convolution is a **learned** downsampling — the network can learn the best way to reduce spatial resolution for the specific task. Modern architectures (ResNet v2, MobileNetV2) prefer stride-2 conv because it's differentiable and gives the model more flexibility. The trade-off: slightly more parameters.

---

**Q3: What are 1×1 convolutions actually computing?**  
> A 1×1 conv at position $(i,j)$ takes the vector of $C_{in}$ values (one per channel at that spatial location) and linearly projects it to $C_{out}$ values. It's equivalent to a dense layer applied **independently to each spatial position**. It can reduce or increase channels and adds non-linearity (with activation). It does NOT mix information between neighbouring pixels — only across channels at one location.

---

**Q4: How do dilated convolutions expand the receptive field without adding parameters?**  
> By inserting holes (zeros) between kernel elements. A 3×3 kernel with dilation $D$ covers a $(2D+1) \times (2D+1)$ area but still only has 9 learnable weights. The formula: effective kernel size = $D(K-1)+1$. Stacking layers with doubling dilation [1,2,4,8] gives a receptive field of $1+2(1+2+4+8)=31$ with only 4 small kernels. Used in WaveNet, DeepLab.

---

**Q5: Why does Global Average Pooling (GAP) replace Flatten+Dense in modern CNNs?**  
> 1. **Parameters:** Flatten+Dense adds $H \times W \times C \times n_{classes}$ params. For VGG's last layer: $7 \times 7 \times 512 \times 4096 = 102M$ params. GAP adds 0.  
> 2. **Structural regularisation:** GAP forces each feature map to encode a globally useful signal, preventing overfitting to spatial location.  
> 3. **Resolution-agnostic:** GAP works on *any* input size — the model can accept variable-resolution images.

---

**Q6: What is the vanishing gradient problem in deep CNNs, and how does BatchNorm help?**  
> As gradients flow backward through many layers, they can shrink exponentially (if activations are in the saturating region of sigmoid/tanh). BatchNorm normalises the pre-activation values to unit Gaussian, keeping them in the most gradient-active region. It also allows higher learning rates. Modern CNNs use ReLU (not tanh) + BatchNorm, making very deep networks (50-200 layers) trainable.

---

**Q7: What is translation equivariance vs translation invariance, and which does a CNN provide?**  
> - **Equivariance:** If the input shifts, the output shifts by the same amount. CNNs are equivariant — a cat ear shifted 10 pixels produces a feature map shifted by 10 pixels.  
> - **Invariance:** If the input shifts, the output stays the same. Pooling adds local invariance — small shifts in the input produce the same pooled output.  
> CNNs provide **equivariance** in conv layers and **local invariance** via pooling. Fully global invariance requires GAP or explicit augmentation.
